<a href="https://colab.research.google.com/github/sharvani1357/FineTuning-LLM/blob/main/FineTuning_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required libraries

!pip install -q transformers
!pip install -q datasets
!pip install -q peft
!pip install -q accelerate
!pip install -q bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.4 MB/s eta 0:00:00


In [ ]:
import transformers
import datasets
import peft
import accelerate
import bitsandbytes as bnb

print("Transformers Version :", transformers.__version__)
print("Datasets Version     :", datasets.__version__)
print("PEFT Version         :", peft.__version__)
print("Accelerate Version   :", accelerate.__version__)
print("BitsAndBytes Loaded Successfully")

Transformers Version : 5.10.1
Datasets Version     : 4.0.0
PEFT Version         : 0.19.1
Accelerate Version   : 1.13.0
BitsAndBytes Loaded Successfully


In [ ]:
import torch

print("CUDA Available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name :", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Enable GPU from Runtime → Change runtime type → GPU")

CUDA Available : True
GPU Name : Tesla T4


In [ ]:
# ============================================================
# TASK 2: LOAD A PRETRAINED TINYLLAMA MODEL
# ============================================================

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# TinyLlama model from Hugging Face
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("\nModel Loaded Successfully!")
print("=" * 50)
print(f"Model Name: {MODEL_NAME}")
print(f"Vocabulary Size: {tokenizer.vocab_size}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ============================================================
# TEST THE MODEL
# ============================================================

prompt = "What is diabetes?"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.7,
    do_sample=True
)

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("\n" + "=" * 50)
print("MODEL RESPONSE")
print("=" * 50)
print(response)

Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


Model Loaded Successfully!
Model Name: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Vocabulary Size: 32000
CUDA Available: True
GPU: Tesla T4


[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



MODEL RESPONSE
What is diabetes?
12. What is the difference between type 1 and type 2 diabetes?
13. How does diabetes affect the body's cells and tissues?
14. How does diabetes affect the circulation and nerve function?
15. What are some lifestyle changes that can help to manage diabetes?
16. How does insulin work in the body to regulate blood sugar levels?
17


In [ ]:
# ============================================================
# TASK 3: ANALYZE MODEL PARAMETERS
# ============================================================

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load TinyLlama
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

# ------------------------------------------------------------
# Calculate Parameters
# ------------------------------------------------------------

total_params = sum(p.numel() for p in model.parameters())

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

trainable_percent = 100 * trainable_params / total_params

# ------------------------------------------------------------
# Display Results
# ------------------------------------------------------------

print("=" * 60)
print("TINYLLAMA PARAMETER ANALYSIS")
print("=" * 60)

print(f"Total Parameters      : {total_params:,}")
print(f"Trainable Parameters  : {trainable_params:,}")
print(f"Trainable Percentage  : {trainable_percent:.4f}%")

print("\n" + "=" * 60)
print("MEMORY ESTIMATION")
print("=" * 60)

# FP16 uses 2 bytes per parameter
memory_gb = (total_params * 2) / (1024**3)

print(f"Approx Model Size (FP16): {memory_gb:.2f} GB")

print("\n" + "=" * 60)
print("WHY FULL FINE-TUNING IS EXPENSIVE")
print("=" * 60)

print("""
1. Every model parameter must be updated.
2. Gradients are stored for all parameters.
3. GPU memory usage becomes very high.
4. Training time increases significantly.
5. Large models may require multiple GPUs.

LoRA solves this by training only a small subset
of parameters while keeping the original model frozen.
""")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

TINYLLAMA PARAMETER ANALYSIS
Total Parameters      : 1,100,048,384
Trainable Parameters  : 1,100,048,384
Trainable Percentage  : 100.0000%

MEMORY ESTIMATION
Approx Model Size (FP16): 2.05 GB

WHY FULL FINE-TUNING IS EXPENSIVE

1. Every model parameter must be updated.
2. Gradients are stored for all parameters.
3. GPU memory usage becomes very high.
4. Training time increases significantly.
5. Large models may require multiple GPUs.

LoRA solves this by training only a small subset
of parameters while keeping the original model frozen.



In [ ]:
# Fix PEFT / TorchAO compatibility issue

!pip uninstall -y torchao
!pip install -q -U torchao

# Restart runtime after installation
import os
os.kill(os.getpid(), 9)

Found existing installation: torchao 0.17.0
Uninstalling torchao-0.17.0:
  Successfully uninstalled torchao-0.17.0


In [ ]:
# ============================================================
# TASK 4 + TASK 5
# APPLY LORA AND VERIFY TRAINABLE PARAMETERS
# ============================================================

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
import torch

# ------------------------------------------------------------
# Load TinyLlama
# ------------------------------------------------------------

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading TinyLlama...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

# ------------------------------------------------------------
# BEFORE LORA
# ------------------------------------------------------------

before_total_params = sum(
    p.numel() for p in model.parameters()
)

before_trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

before_percentage = (
    100 * before_trainable_params / before_total_params
)

print("\n" + "=" * 60)
print("BEFORE APPLYING LORA")
print("=" * 60)

print(f"Total Parameters      : {before_total_params:,}")
print(f"Trainable Parameters  : {before_trainable_params:,}")
print(f"Trainable Percentage  : {before_percentage:.4f}%")

# ------------------------------------------------------------
# TASK 4 : APPLY LORA
# ------------------------------------------------------------

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

model = get_peft_model(
    model,
    lora_config
)

# ------------------------------------------------------------
# AFTER LORA
# ------------------------------------------------------------

after_total_params = sum(
    p.numel() for p in model.parameters()
)

after_trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

after_percentage = (
    100 * after_trainable_params / after_total_params
)

# ------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("AFTER APPLYING LORA")
print("=" * 60)

print(f"Total Parameters      : {after_total_params:,}")
print(f"Trainable Parameters  : {after_trainable_params:,}")
print(f"Trainable Percentage  : {after_percentage:.6f}%")

# ------------------------------------------------------------
# PARAMETER REDUCTION
# ------------------------------------------------------------

reduction = (
    100
    - (after_trainable_params / before_trainable_params) * 100
)

print("\n" + "=" * 60)
print("LORA PARAMETER EFFICIENCY")
print("=" * 60)

print(
    f"Reduction in Trainable Parameters : "
    f"{reduction:.2f}%"
)

# ------------------------------------------------------------
# TRAINABLE LAYERS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("TRAINABLE LORA LAYERS")
print("=" * 60)

for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)

# ------------------------------------------------------------
# PEFT SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("PEFT MODEL SUMMARY")
print("=" * 60)

model.print_trainable_parameters()

Loading TinyLlama...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


BEFORE APPLYING LORA
Total Parameters      : 1,100,048,384
Trainable Parameters  : 1,100,048,384
Trainable Percentage  : 100.0000%

AFTER APPLYING LORA
Total Parameters      : 1,102,301,184
Trainable Parameters  : 2,252,800
Trainable Percentage  : 0.204372%

LORA PARAMETER EFFICIENCY
Reduction in Trainable Parameters : 99.80%

TRAINABLE LORA LAYERS
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight
base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight
base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight
base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight
base_mode

In [ ]:
# ============================================================
# TASK 6: CREATE MEDICAL QA DATASET
# ============================================================

from datasets import Dataset
import pandas as pd

# Medical QA Dataset

data = {
    "question": [
        "What is diabetes?",
        "How to treat a common cold?",
        "What are the symptoms of hypertension?"
    ],
    "answer": [
        "Diabetes is a chronic disease where the body cannot regulate blood sugar properly.",
        "Rest, hydration, and over-the-counter medications can help manage symptoms.",
        "Symptoms include headaches, shortness of breath, and nosebleeds."
    ]
}

# Convert to DataFrame

df = pd.DataFrame(data)

# Convert to Hugging Face Dataset

dataset = Dataset.from_pandas(df)

print("Dataset Created Successfully\n")

print(dataset)

print("\nSample Records:\n")

for i in range(len(dataset)):
    print(f"Question: {dataset[i]['question']}")
    print(f"Answer: {dataset[i]['answer']}")
    print("-" * 60)

Dataset Created Successfully

Dataset({
    features: ['question', 'answer'],
    num_rows: 3
})

Sample Records:

Question: What is diabetes?
Answer: Diabetes is a chronic disease where the body cannot regulate blood sugar properly.
------------------------------------------------------------
Question: How to treat a common cold?
Answer: Rest, hydration, and over-the-counter medications can help manage symptoms.
------------------------------------------------------------
Question: What are the symptoms of hypertension?
Answer: Symptoms include headaches, shortness of breath, and nosebleeds.
------------------------------------------------------------


In [ ]:
# ============================================================
# TASK 7: TOKENIZATION
# ============================================================

from transformers import AutoTokenizer

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# TinyLlama needs a pad token

tokenizer.pad_token = tokenizer.eos_token

# Dataset

data = {
    "question": [
        "What is diabetes?",
        "How to treat a common cold?",
        "What are the symptoms of hypertension?"
    ],
    "answer": [
        "Diabetes is a chronic disease where the body cannot regulate blood sugar properly.",
        "Rest, hydration, and over-the-counter medications can help manage symptoms.",
        "Symptoms include headaches, shortness of breath, and nosebleeds."
    ]
}

texts = [
    f"Question: {q}\nAnswer: {a}"
    for q, a in zip(data["question"], data["answer"])
]

# Tokenization

encodings = tokenizer(
    texts,
    truncation=True,
    padding=True,
    max_length=128
)

print("INPUT IDS\n")
print(encodings["input_ids"])

print("\n" + "="*80)

print("\nATTENTION MASKS\n")
print(encodings["attention_mask"])

INPUT IDS

[[1, 894, 29901, 1724, 338, 652, 370, 10778, 29973, 13, 22550, 29901, 4671, 370, 10778, 338, 263, 17168, 293, 17135, 988, 278, 3573, 2609, 1072, 5987, 10416, 26438, 6284, 29889, 2, 2, 2], [1, 894, 29901, 1128, 304, 7539, 263, 3619, 11220, 29973, 13, 22550, 29901, 11654, 29892, 27246, 29878, 362, 29892, 322, 975, 29899, 1552, 29899, 11808, 13589, 800, 508, 1371, 10933, 25828, 4835, 29889], [1, 894, 29901, 1724, 526, 278, 25828, 4835, 310, 7498, 10700, 2673, 29973, 13, 22550, 29901, 10667, 415, 4835, 3160, 2343, 14520, 29892, 3273, 2264, 310, 16172, 29892, 322, 26414, 569, 5779, 29889]]


ATTENTION MASKS

[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]


In [ ]:
# ============================================================
# TASK 8: FINE-TUNE USING LORA
# ============================================================

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

from datasets import Dataset
import pandas as pd
import torch

# ------------------------------------------------------------
# Load Model
# ------------------------------------------------------------

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

# ------------------------------------------------------------
# Apply LoRA
# ------------------------------------------------------------

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj","k_proj","v_proj","o_proj"]
)

model = get_peft_model(model, lora_config)

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------

data = {
    "question": [
        "What is diabetes?",
        "How to treat a common cold?",
        "What are the symptoms of hypertension?"
    ],
    "answer": [
        "Diabetes is a chronic disease where the body cannot regulate blood sugar properly.",
        "Rest, hydration, and over-the-counter medications can help manage symptoms.",
        "Symptoms include headaches, shortness of breath, and nosebleeds."
    ]
}

df = pd.DataFrame(data)

dataset = Dataset.from_pandas(df)

# ------------------------------------------------------------
# Formatting
# ------------------------------------------------------------

def format_examples(example):
    text = f"Question: {example['question']}\nAnswer: {example['answer']}"
    tokens = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=128
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

tokenized_dataset = dataset.map(format_examples)

# ------------------------------------------------------------
# Training Arguments
# ------------------------------------------------------------

training_args = TrainingArguments(
    output_dir="./medical_lora_output",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

# ------------------------------------------------------------
# Trainer
# ------------------------------------------------------------

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    )
)

# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

trainer.train()

print("\nLoRA Fine-Tuning Completed Successfully!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Step,Training Loss
1,1.789447
2,1.805073
3,1.904980
4,1.882103
5,1.700011
6,1.730963
7,1.826142
8,1.698769
9,1.646268
10,1.672819



LoRA Fine-Tuning Completed Successfully!


In [ ]:
# ============================================================
# TASK 9: SAVE LORA ADAPTER
# ============================================================

# Save only LoRA adapter weights

save_path = "./medical_lora_adapter"

model.save_pretrained(save_path)

tokenizer.save_pretrained(save_path)

print("LoRA Adapter Saved Successfully!")

print(f"\nSaved Location: {save_path}")

# Display files

import os

print("\nSaved Files:\n")

for file in os.listdir(save_path):
    print(file)

LoRA Adapter Saved Successfully!

Saved Location: ./medical_lora_adapter

Saved Files:

chat_template.jinja
adapter_config.json
tokenizer.json
adapter_model.safetensors
tokenizer_config.json
README.md


In [ ]:
# ============================================================
# TASK 10: MEDICAL QUESTION ANSWERING
# COMPARE BEFORE LORA VS AFTER LORA
# ============================================================

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
ADAPTER_PATH = "./medical_lora_adapter"

# ------------------------------------------------------------
# LOAD TOKENIZER
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ============================================================
# BEFORE LORA
# ============================================================

print("=" * 70)
print("BEFORE LORA (BASE MODEL)")
print("=" * 70)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

question = "What is diabetes?"

prompt = f"Question: {question}\nAnswer:"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(base_model.device)

outputs = base_model.generate(
    **inputs,
    max_new_tokens=80,
    temperature=0.7,
    do_sample=True
)

base_response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("\nQuestion:")
print(question)

print("\nResponse:")
print(base_response)

# ============================================================
# AFTER LORA
# ============================================================

print("\n")
print("=" * 70)
print("AFTER LORA (MEDICAL MODEL)")
print("=" * 70)

base_model_for_lora = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

lora_model = PeftModel.from_pretrained(
    base_model_for_lora,
    ADAPTER_PATH
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(lora_model.device)

outputs = lora_model.generate(
    **inputs,
    max_new_tokens=80,
    temperature=0.7,
    do_sample=True
)

lora_response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("\nQuestion:")
print(question)

print("\nResponse:")
print(lora_response)

# ============================================================
# COMPARISON
# ============================================================

print("\n")
print("=" * 70)
print("EVALUATION")
print("=" * 70)

print("\nQuestion:")
print(question)

print("\nBEFORE LoRA (Generic Response):")
print(base_response)

print("\nAFTER LoRA (Medical Domain Response):")
print(lora_response)

print("\n")
print("=" * 70)
print("COMPARISON COMPLETED")
print("=" * 70)

BEFORE LORA (BASE MODEL)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question:
What is diabetes?

Response:
Question: What is diabetes?
Answer: Diabetes is a chronic metabolic disorder characterized by increased blood glucose levels, failure to regulate blood glucose levels, and damage to tissues caused by high blood sugar levels.
Question: What are the symptoms of diabetes?
Answer: The symptoms of diabetes can include blurry vision, fatigue, muscle pain


AFTER LORA (MEDICAL MODEL)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question:
What is diabetes?

Response:
Question: What is diabetes?
Answer: Diabetes is a metabolic disorder in which the body is unable to produce enough insulin, a hormone that enables sugar to be broken down into glucose, or glucose to be used by cells.

Based on the text material, what is the typical course of diabetes, and what are some potential complications of the disease?


EVALUATION

Question:
What is diabetes?

BEFORE LoRA (Generic Response):
Question: What is diabetes?
Answer: Diabetes is a chronic metabolic disorder characterized by increased blood glucose levels, failure to regulate blood glucose levels, and damage to tissues caused by high blood sugar levels.
Question: What are the symptoms of diabetes?
Answer: The symptoms of diabetes can include blurry vision, fatigue, muscle pain

AFTER LoRA (Medical Domain Response):
Question: What is diabetes?
Answer: Diabetes is a metabolic disorder in which the body is unable to produce enough insulin, a hormone that enables suga